In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [5]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
 	print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")


Input count: 4
Screenshot 2026-11-08 134946.png bytes: 58752
Screenshot 2026-11-08 135129.png bytes: 74365
Screenshot 2026-11-08 135201.png bytes: 68415
Screenshot 2026-11-08 135323.png bytes: 85798
Upload verified. Select your relevant input(s) for Step A3.


In [3]:
# Cell 3: Process and standardize images
from PIL import Image
import hashlib

output_folder = PROCESSED_DIR / "images"
output_folder.mkdir(parents=True, exist_ok=True)

for img_path in (RAW_DIR / "images").glob("*"):
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")           # Fix color channels
            img = img.resize((224, 224))        # Standardize size

            # Save cleanly
            target = output_folder / f"clean_{img_path.name}"
            img.save(target)
            print(f"Successfully processed: {img_path.name} -> {target.name}")
    except Exception as e:
        print(f"Error on {img_path.name}: {e}")

In [6]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")


Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [7]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))


Common lineage registry ready.
Registered entries: 0


In [ ]:
# # Cell 5A1
# # PATHWAY A1 — PDF/TXT
# # CONDITIONAL REQUIRED: Run only for document-based projects
# # Install PDF processing library
# !pip -q install pypdf

# from pypdf import PdfReader
# import re

# # Step 1: Select the exact source file
# # Example: Synthetic practice PDF
# # Replace with your actual project file where applicable.
# f = select_source(
#     relative_path=(
#         "topic3_dummy_datasets/"
#         "01_documents/"
#         "student_handbook_demo.pdf"
#     )
# )

# # Step 2: Extract text according to file type
# if f.suffix.lower() == ".pdf":
#     reader = PdfReader(str(f))
#     pages = [
#         (i + 1, page.extract_text() or "")
#         for i, page in enumerate(reader.pages)
#     ]
# elif f.suffix.lower() == ".txt":
#     pages = [(1, f.read_text(encoding="utf-8"))]
# else:
#     raise ValueError("Pathway A needs a PDF or TXT source")
# # Step 3: Basic text cleaning
# text = "\n".join(content for _, content in pages)
# clean = re.sub(r"[ \t]+", " ", text).strip()

# print("Source:", f.name)
# print("Pages/segments:", len(pages))
# print("Extracted characters:", len(text))
# print("Cleaned characters:", len(clean))
# print("Content sample:", repr(clean[:300]))

# assert clean, "No usable text: check extraction or OCR needs."
# # Step 4: Fixed-size chunking
# def fixed_chunks(s, size=500):
#     if size <= 0:
#         raise ValueError("Chunk size must be positive")
#     return [
#         s[i:i + size]
#         for i in range(0, len(s), size)
#     ]
# # Step 5: Paragraph-aware chunking
# def paragraph_chunks(s, size=500):
#     if size <= 0:
#         raise ValueError("Chunk size must be positive")
#     parts = []
#     current = ""
#     for segment in re.split(r"\n\s*\n", s):
#         segment = segment.strip()
#         if not segment:
#             continue
#         if len(segment) > size:
#             if current:
#                 parts.append(current)
#                 current = ""
#             parts.extend(fixed_chunks(segment, size))

#         elif len(current) + len(segment) + (2 if current else 0) <= size:
#             current += ("\n\n" if current else "") + segment
#         else:
#             if current:
#                 parts.append(current)
#             current = segment
#     if current:
#         parts.append(current)
#     return parts
# # Step 6: Compare two preparation approaches
# fixed = fixed_chunks(clean, size=500)
# structured = paragraph_chunks(clean, size=500)

# print("\n--- Chunking Comparison ---")
# print("Fixed chunks:", len(fixed))
# print("Paragraph-aware chunks:", len(structured))

# print("\nFixed sample:")
# print(repr(fixed[0][:180]))

# print("\nParagraph-aware sample:")
# print(repr(structured[0][:180]))

# print("\nPATHWAY A1 completed successfully.")


In [ ]:
# # Cell 5A2
# # PATHWAY A2 — REQUIRED after A1
# # Choose and export traceable text chunks
# import json
# # Step 1: Select the preparation approach
# # Change to "fixed" if your Part A1 evidence supports it.
# CHOSEN_METHOD = "paragraph-aware"
# if CHOSEN_METHOD == "paragraph-aware":
#     chosen = structured
#     approach = "paragraph-aware-char-500"
# elif CHOSEN_METHOD == "fixed":
#     chosen = fixed
#     approach = "fixed-char-500"
# else:
#     raise ValueError("Choose 'paragraph-aware' or 'fixed'")
# # Step 2: Create traceable records
# records = [
#     {
#         "record_id": f"{f.stem}-{i:04d}",
#         "text": chunk,
#         "source": str(f.relative_to(RAW_DIR)),
#         "source_sha256": source_hash(f),
#         "processing": approach,
#         "page": None
#     }
#     for i, chunk in enumerate(chosen, 1)
# ]
# # Page is None because the basic whole-document
# # splitter does not preserve page boundaries.
# # Step 3: Export prepared records
# prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
# prepared_path.parent.mkdir(parents=True, exist_ok=True)
# with prepared_path.open("w", encoding="utf-8") as output:
#     for row in records:
#         output.write(
#             json.dumps(row, ensure_ascii=False) + "\n"
#         )
# # Step 4: Basic validation
# assert prepared_path.is_file(), "Prepared output not found"
# assert prepared_path.stat().st_size > 0, "Prepared output is empty"
# assert records, "No text chunks were produced"
# assert all(row["text"].strip() for row in records), \
#     "One or more chunks are empty"
# with prepared_path.open("r", encoding="utf-8") as file:
#     exported_records = [
#         json.loads(line)
#         for line in file
#         if line.strip()
#     ]
# assert len(exported_records) == len(records), \
#     "Exported record count does not match"

# # Step 5: Display evidence
# print("PATHWAY A2 completed successfully.")
# print("Selected approach:", approach)
# print("Total chunks exported:", len(records))
# print("Prepared file:", prepared_path)
# print("File size:", prepared_path.stat().st_size, "bytes")
# print("Validation: PASS")
# print("First record:", records[0])


In [ ]:
# # Cell 5B
# # PATHWAY B — CSV/XLSX
# # CONDITIONAL REQUIRED: Run only for tabular-data projects
# import pandas as pd
# # Step 1: Select the exact source file
# # Replace with your actual project data path where applicable.
# f = select_source(
#     relative_path=(
#         "topic3_dummy_datasets/"
#         "02_tabular/"
#         "student_performance_demo.csv"
#     )
# )
# # Step 2: Load the selected dataset
# if f.suffix.lower() == ".csv":
#     df = pd.read_csv(f)
# elif f.suffix.lower() == ".xlsx":
#     df = pd.read_excel(f)
# else:
#     raise ValueError("Select CSV or XLSX for Pathway B")
# # Step 3: Inspect original data
# print("--- Original Dataset ---")
# print("Source:", f.name)
# print("Rows/columns:", df.shape)
# print("Columns:", df.columns.tolist())
# print(df.head().to_string(index=False))
# print("Missing per column:", df.isna().sum().to_dict())
# print("Exact duplicate rows:", int(df.duplicated().sum()))
# assert not df.empty, "The selected dataset is empty."
# # Step 4: Apply justified cleaning rules
# # Change these settings based on your Part B decisions.
# REMOVE_DUPLICATES = True
# TRIM_TEXT = True
# prepared = df.copy()
# if REMOVE_DUPLICATES:
#     prepared = prepared.drop_duplicates().copy()
# if TRIM_TEXT:
#     for col in prepared.select_dtypes(
#         include=["object", "string"]
#     ).columns:
#         prepared[col] = (
#             prepared[col].astype("string").str.strip()
#         )
# # Do NOT automatically impute missing values,
# # change target labels, or remove outliers.
# # Step 5: Compare before and after preparation
# print("\n--- Preparation Comparison ---")
# print("Original rows:", len(df))
# print("Prepared rows:", len(prepared))
# print("Rows removed:", len(df) - len(prepared))
# print("Missing values after:", prepared.isna().sum().to_dict())
# print("\nPrepared data sample:")
# print(prepared.head().to_string(index=False))
# # Step 6: Export prepared data
# prepared_path = PROCESSED_DIR / "prepared_table.csv"
# prepared_path.parent.mkdir(parents=True, exist_ok=True)
# prepared.to_csv(prepared_path, index=False)
# # Step 7: Validate exported data
# assert prepared_path.is_file(), "Prepared output not found"
# assert prepared_path.stat().st_size > 0, "Prepared output is empty"
# assert not prepared.empty, "No prepared data rows remain"
# exported = pd.read_csv(prepared_path)
# assert len(exported) == len(prepared), (
#     "Exported row count does not match prepared data"
# )
# assert exported.columns.tolist() == prepared.columns.tolist(), (
#     "Exported columns do not match prepared data"
# )
# # Step 8: Display evidence
# print("\nPATHWAY B completed successfully.")
# print("Remove duplicates:", REMOVE_DUPLICATES)
# print("Trim text:", TRIM_TEXT)
# print("Prepared shape:", prepared.shape)
# print("Saved:", prepared_path)
# print("Validation: PASS")


In [9]:
# Cell 5C
# PATHWAY C — IMAGE DATA
# CONDITIONAL REQUIRED: Run only for computer-vision projects
from PIL import Image, UnidentifiedImageError
import pandas as pd
# Step 1: Select the source image folder
# Replace with your actual project image folder where applicable.
source_dir = RAW_DIR
assert source_dir.is_dir(), (
    f"Choose an existing image folder: {source_dir}"
)
# Step 2: Collect supported image files
image_files = sorted(
    p for p in source_dir.iterdir()
    if p.is_file() and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".webp"
    }
)
assert image_files, "No images found in the selected folder"
print("--- Source Image Folder ---")
print("Folder:", source_dir)
print("Image count:", len(image_files))
print("Example files:", [p.name for p in image_files[:5]])
# Step 3: Configure preparation settings
# Change these only if justified by your Part B decision.
TARGET_SIZE = (224, 224)
CONVERT_TO_RGB = True
# Step 4: Prepare output folder
image_dir = PROCESSED_DIR / "images"
image_dir.mkdir(parents=True, exist_ok=True)
rows = []
failed_files = []
# Step 5: Process images
for f in image_files:
    try:
        with Image.open(f) as im:
            im.load()
            original_size = im.size
            original_mode = im.mode
            transformed = im
            if CONVERT_TO_RGB:
                transformed = transformed.convert("RGB")
            transformed = transformed.resize(TARGET_SIZE)
            target = image_dir / f"{f.stem}_{source_hash(f)[:12]}.png"
            transformed.save(target)
        rows.append({
            "source": str(f.relative_to(RAW_DIR)),
            "source_sha256": source_hash(f),
            "original_width": original_size[0],
            "original_height": original_size[1],
            "original_mode": original_mode,
            "output": str(target.relative_to(BASE_DIR)),
            "output_width": TARGET_SIZE[0],
            "output_height": TARGET_SIZE[1],
            "processing": (
                f"convert_to_rgb={CONVERT_TO_RGB}; "
                f"resize={TARGET_SIZE}"
            )
        })
    except (UnidentifiedImageError, OSError) as err:
        failed_files.append({
            "file": f.name,
            "error": str(err)
        })
        print("Failed:", f.name, "|", str(err))
# Step 6: Validate processing results
assert rows, "No readable images were prepared"
prepared_path = PROCESSED_DIR / "image_manifest.csv"
pd.DataFrame(rows).to_csv(prepared_path, index=False)
assert prepared_path.is_file(), "Image manifest was not created"
assert prepared_path.stat().st_size > 0, "Image manifest is empty"
# Step 7: Show evidence
print("\n--- Preparation Summary ---")
print("Images prepared:", len(rows))
print("Images failed:", len(failed_files))
print("Prepared image folder:", image_dir)
print("Manifest:", prepared_path)
manifest_df = pd.read_csv(prepared_path)
print("\nManifest preview:")
print(manifest_df.head().to_string(index=False))
# Step 8: Important note for labels
print("\nNote:")
print(
    "If your project uses labels or annotations "
    "(for example labels.csv, bounding boxes, or masks), "
    "they remain raw unless you explicitly process them."
)
print(
    "Do not infer labels from filenames unless this is "
    "justified by your dataset design."
)
# Preserve the image folder reference for Step B4
f = source_dir
print("\nPATHWAY C completed successfully.")

--- Source Image Folder ---
Folder: /content/week3_project/data/raw
Image count: 4
Example files: ['Screenshot 2026-11-08 134946.png', 'Screenshot 2026-11-08 135129.png', 'Screenshot 2026-11-08 135201.png', 'Screenshot 2026-11-08 135323.png']

--- Preparation Summary ---
Images prepared: 4
Images failed: 0
Prepared image folder: /content/week3_project/data/processed/images
Manifest: /content/week3_project/data/processed/image_manifest.csv

Manifest preview:
                          source                                                    source_sha256  original_width  original_height original_mode                                                              output  output_width  output_height                             processing
Screenshot 2026-11-08 134946.png 338561e11f39b0136c5bbbfbfbf413e5b48525d4acaa4b90afc8131bd78d39ee             221              347          RGBA data/processed/images/Screenshot 2026-11-08 134946_338561e11f39.png           224            224 convert_to_rgb=

In [ ]:
# # Cell 5D
# # PATHWAY D — JSON/API EXPORT
# # CONDITIONAL REQUIRED: Run only for JSON/API projects
# import json
# # Step 1: Select the exact JSON source
# # Replace this path with your actual dataset where applicable.
# f = select_source(
#     relative_path=(
#         "topic3_dummy_datasets/"
#         "04_json_api/"
#         "products_api_demo.json"
#     )
# )
# # Step 2: Load the JSON source
# with f.open("r", encoding="utf-8") as file:
#     raw = json.load(file)
# # Step 3: Configure record extraction and required fields
# # These values match the supplied synthetic example.
# # Change them to match your actual project schema.
# RECORD_KEY = "items"
# REQUIRED_KEYS = ["item_id"]
# # Step 4: Extract JSON records
# if isinstance(raw, list):
#     items = raw
# elif isinstance(raw, dict) and isinstance(raw.get(RECORD_KEY), list):
#     items = raw[RECORD_KEY]
# elif isinstance(raw, dict) and RECORD_KEY not in raw:
#     items = [raw]
# else:
#     raise ValueError(
#         "Expected a JSON list, a JSON object, or "
#         f"an object containing a list under '{RECORD_KEY}'."
#     )
# assert items, "No JSON records were found."
# # Step 5: Inspect record structure
# print("--- Original JSON Data ---")
# print("Source:", f.name)
# print("Top-level format:", type(raw).__name__)
# print("Record count:", len(items))
# # Separate object records from invalid record types
# object_records = [
#     item for item in items
#     if isinstance(item, dict)
# ]
# invalid_types = len(items) - len(object_records)
# if object_records:
#     print("First record keys:", list(object_records[0].keys()))
#     print("First record:", object_records[0])
# print("Records with invalid types:", invalid_types)
# # Step 6: Validate required fields
# # This checks field presence, not the validity of field values.
# valid = [
#     item for item in object_records
#     if all(key in item for key in REQUIRED_KEYS)
# ]
# missing_required = len(object_records) - len(valid)
# print("\n--- Validation Summary ---")
# print("Original records:", len(items))
# print("Valid records:", len(valid))
# print("Records with invalid types:", invalid_types)
# print("Records missing required keys:", missing_required)
# assert valid, "No valid JSON records remain after validation."
# # Step 7: Export valid records to JSONL
# prepared_path = PROCESSED_DIR / "prepared_json.jsonl"
# prepared_path.parent.mkdir(parents=True, exist_ok=True)
# with prepared_path.open("w", encoding="utf-8") as output:
#     for row in valid:
#         output.write(
#             json.dumps(row, ensure_ascii=False) + "\n"
#         )
# # Step 8: Validate the exported JSONL
# assert prepared_path.is_file(), "Prepared output not found"
# assert prepared_path.stat().st_size > 0, "Prepared output is empty"
# with prepared_path.open("r", encoding="utf-8") as file:
#     exported_records = [
#         json.loads(line)
#         for line in file
#         if line.strip()
#     ]
# assert len(exported_records) == len(valid), (
#     "Exported record count does not match"
# )
# assert exported_records == valid, (
#     "Exported content differs from prepared records"
# )
# # Step 9: Display evidence
# print("\nPATHWAY D completed successfully.")
# print("Required keys:", REQUIRED_KEYS)
# print("Exported records:", len(exported_records))
# print("Saved:", prepared_path)
# print("Validation: PASS")


In [ ]:
# # Cell 5E
# # PATHWAY E — AUDIO INSPECTION
# # CONDITIONAL REQUIRED: Run only for audio-based projects
# # Step 1: Install and import the audio library
# !pip -q install soundfile
# import soundfile as sf
# import json
# # Step 2: Select the exact source file
# # Replace this path with your own project audio file if applicable.
# f = select_source(
#     relative_path=(
#         "topic3_dummy_datasets/"
#         "05_audio/"
#         "tone_low.wav"
#     )
# )
# # Step 3: Inspect audio metadata
# info = sf.info(str(f))
# print("--- Original Audio Source ---")
# print("Selected file:", f.name)
# print("Sample rate:", info.samplerate, "Hz")
# print("Channels:", info.channels)
# print("Duration:", info.duration, "seconds")
# print("Frames:", info.frames)
# print("Format:", info.format)
# print("Subtype:", info.subtype)
# # Step 4: Basic source validation
# assert info.frames > 0, "Audio source has no frames"
# assert info.samplerate > 0, "Invalid sample rate"
# assert info.channels > 0, "Invalid channel count"
# assert info.duration > 0, "Invalid audio duration"
# # No audio transformation is performed.
# # Resampling, trimming or feature extraction must be
# # justified by the intended AI task.
# # Step 5: Create audio inventory with provenance
# audio_record = {
#     "source": str(f.relative_to(RAW_DIR)),
#     "source_sha256": source_hash(f),
#     "sample_rate": info.samplerate,
#     "channels": info.channels,
#     "duration_seconds": info.duration,
#     "frames": info.frames,
#     "format": info.format,
#     "subtype": info.subtype,
#     "processing": "inspection only; source retained"
# }
# # Step 6: Export inventory
# prepared_path = PROCESSED_DIR / "audio_inventory.json"
# prepared_path.parent.mkdir(parents=True, exist_ok=True)
# prepared_path.write_text(
#     json.dumps(audio_record, indent=2, ensure_ascii=False),
#     encoding="utf-8"
# )
# # Step 7: Validate exported inventory
# assert prepared_path.is_file(), "Audio inventory not found"
# assert prepared_path.stat().st_size > 0, "Audio inventory is empty"
# with prepared_path.open("r", encoding="utf-8") as file:
#     exported_record = json.load(file)
# assert exported_record == audio_record, (
#     "Exported metadata does not match inspected audio"
# )
# # Step 8: Display evidence
# print("\nPATHWAY E completed successfully.")
# print("Processing: Inspection only")
# print("Original audio: Unchanged")
# print("Saved inventory:", prepared_path)
# print("Validation: PASS")


In [ ]:
# # Cell 5F
# # PATHWAY F — TIME-SERIES
# # CONDITIONAL REQUIRED: Run only for time-series projects
# import pandas as pd
# # Step 1: Select the exact source file
# # Replace with your actual project dataset where applicable.
# f = select_source(
#     relative_path=(
#         "topic3_dummy_datasets/"
#         "06_timeseries/"
#         "energy_readings_demo.csv"
#     )
# )
# # Step 2: Load the selected dataset
# df = pd.read_csv(f)
# print("--- Original Time-Series Dataset ---")
# print("Selected source:", str(f.relative_to(RAW_DIR)))
# print("Rows/columns:", df.shape)
# print("Columns:", df.columns.tolist())
# print(df.head().to_string(index=False))
# assert not df.empty, "The selected dataset is empty."
# # Step 3: Configure the timestamp column
# # Replace with your actual timestamp column name.
# TIME_COLUMN = "timestamp"
# DROP_INVALID_TIMESTAMPS = True
# if TIME_COLUMN not in df.columns:
#     raise ValueError(
#         f"Expected time column {TIME_COLUMN!r}; "
#         f"available: {df.columns.tolist()}"
#     )
# # Step 4: Parse and inspect timestamps
# prepared = df.copy()
# prepared[TIME_COLUMN] = pd.to_datetime(
#     prepared[TIME_COLUMN],
#     errors="coerce",
#     utc=True
# )
# invalid_timestamps = int(
#     prepared[TIME_COLUMN].isna().sum()
# )
# print("\n--- Timestamp Inspection ---")
# print("Original rows:", len(df))
# print("Invalid timestamps:", invalid_timestamps)
# print(
#     "Duplicate valid timestamps:",
#     int(prepared[TIME_COLUMN].dropna().duplicated().sum())
# )
# # Step 5: Apply the selected preparation decision
# # Keep missing sensor readings for Topic 4 quality review.
# if DROP_INVALID_TIMESTAMPS:
#     prepared = prepared.dropna(
#         subset=[TIME_COLUMN]
#     ).copy()
# prepared = prepared.sort_values(
#     TIME_COLUMN,
#     na_position="last"
# ).copy()

# assert not prepared.empty, (
#     "No records remain after time-series preparation."
# )
# # Step 6: Compare before and after
# print("\n--- Preparation Comparison ---")
# print("Original rows:", len(df))
# print("Prepared rows:", len(prepared))
# print("Rows removed:", len(df) - len(prepared))
# print("Missing values:", prepared.isna().sum().to_dict())
# print("\nPrepared data sample:")
# print(prepared.head().to_string(index=False))
# # Step 7: Export prepared dataset
# prepared_path = PROCESSED_DIR / "prepared_timeseries.csv"
# prepared_path.parent.mkdir(parents=True, exist_ok=True)
# prepared.to_csv(prepared_path, index=False)
# # Step 8: Validate exported data
# assert prepared_path.is_file(), "Prepared output not found"
# assert prepared_path.stat().st_size > 0, "Prepared output is empty"
# exported = pd.read_csv(prepared_path)
# assert len(exported) == len(prepared), (
#     "Exported row count does not match prepared data"
# )
# assert exported.columns.tolist() == prepared.columns.tolist(), (
#     "Exported columns do not match prepared data"
# )
# # Check chronological ordering of valid timestamps
# valid_exported_times = pd.to_datetime(
#     exported[TIME_COLUMN],
#     errors="coerce",
#     utc=True
# ).dropna()
# assert valid_exported_times.is_monotonic_increasing, (
#     "Exported timestamps are not chronologically sorted"
# )
# print("\nPATHWAY F completed successfully.")
# print("Timestamp column:", TIME_COLUMN)
# print("Drop invalid timestamps:", DROP_INVALID_TIMESTAMPS)
# print("Prepared rows:", len(prepared))
# print("Saved:", prepared_path)
# print("Validation: PASS")


In [10]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')


Output: /content/week3_project/data/processed/image_manifest.csv Size: 1023
Validation: file exists and is non-empty; verify content manually too.


In [15]:
# # Register Pathway A — PDF/TXT
# register_lineage(
#     source=f,
#     output=prepared_path,
#     processing=approach
# )


In [14]:
# # Register Pathway B — CSV/XLSX
# register_lineage(
#     source=f,
#     output=prepared_path,
#     processing=(
#         f"remove_duplicates={REMOVE_DUPLICATES}; "
#         f"trim_text={TRIM_TEXT}"
#     )
# )


In [13]:
# Register Pathway C — Images
for row in rows:
    register_lineage(
        source=RAW_DIR / row["source"],
        output=BASE_DIR / row["output"],
        processing=row["processing"]
    )



Registered: Screenshot 2026-11-08 134946.png -> Screenshot 2026-11-08 134946_338561e11f39.png
Registered: Screenshot 2026-11-08 135129.png -> Screenshot 2026-11-08 135129_0c149a0dd48a.png
Registered: Screenshot 2026-11-08 135201.png -> Screenshot 2026-11-08 135201_5a04dea994dd.png
Registered: Screenshot 2026-11-08 135323.png -> Screenshot 2026-11-08 135323_a8a57dea156c.png


In [16]:
# # Register Pathway D — JSON/API
# register_lineage(
#     source=f,
#     output=prepared_path,
#     processing=(
#         f"JSON to JSONL; "
#         f"record_key={RECORD_KEY}; "
#         f"required_keys={REQUIRED_KEYS}; "
#         "export valid records"
#     )
# )



In [17]:
# # Register Pathway E — Audio
# register_lineage(
#     source=f,
#     output=prepared_path,
#     processing=(
#         "Audio metadata inspection only; "
#         "original audio unchanged"
#     )
# )


In [18]:
# # Register Pathway F — Time-Series
# register_lineage(
#     source=f,
#     output=prepared_path,
#     processing=(
#         f"time_column={TIME_COLUMN}; "
#         f"drop_invalid_timestamps="
#         f"{DROP_INVALID_TIMESTAMPS}; "
#         "chronological_sort=True"
#     )
# )


In [19]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)



Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 4
Saved to: /content/week3_project/data/week3_manifest.json


In [20]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)


COMMON LINEAGE VALIDATION: PASS
Verified records: 4
Manifest: /content/week3_project/data/week3_manifest.json


In [21]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")


Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [23]:
# # Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# # Run only if your project uses document data.
# # Requires Cell 7 (Common V1 Pipeline Utilities).

# !pip -q install pypdf

# import re
# import json
# from pypdf import PdfReader

# # Step 1: Select the source and preparation configuration
# # Replace with your actual source and Part B decisions.

# SOURCE_A = resolve_source(
#     "topic3_dummy_datasets/01_documents/student_handbook_demo.pdf"
# )

# CONFIG_A = {
#     "chunk_size": 500,
#     "method": "paragraph-aware"
# }

# # Step 2: Inspect document content
# def inspect_documents(inputs, cfg):
#     src = inputs[0]

#     if src.suffix.lower() == ".pdf":
#         reader = PdfReader(str(src))
#         pages = [
#             page.extract_text() or ""
#             for page in reader.pages
#         ]
#         text = "\n".join(pages)
#         page_count = len(pages)

#     elif src.suffix.lower() == ".txt":
#         text = src.read_text(encoding="utf-8")
#         page_count = None

#     else:
#         raise ValueError("Expected a PDF or TXT source")

#     if not text.strip():
#         raise ValueError(
#             "No extractable text found. "
#             "Scanned PDFs may require OCR."
#         )

#     return {
#         "source": src.name,
#         "characters": len(text),
#         "pages": page_count
#     }


# # Step 3: Define document preparation
# def prepare_documents(inputs, out, cfg):
#     src = inputs[0]

#     # Extract text
#     if src.suffix.lower() == ".pdf":
#         reader = PdfReader(str(src))
#         text = "\n".join(
#             page.extract_text() or ""
#             for page in reader.pages
#         )

#     elif src.suffix.lower() == ".txt":
#         text = src.read_text(encoding="utf-8")

#     else:
#         raise ValueError("Expected a PDF or TXT source")

#     # Apply the same cleaning used in Part B
#     text = re.sub(r"[ \t]+", " ", text).strip()

#     size = int(cfg["chunk_size"])
#     method = cfg["method"]

#     if size <= 0:
#         raise ValueError("Chunk size must be positive")

#     # Fixed-size chunking
#     if method == "fixed":
#         chunks = [
#             text[i:i + size]
#             for i in range(0, len(text), size)
#         ]

#     # Paragraph-aware chunking
#     elif method == "paragraph-aware":
#         chunks = []
#         current = ""

#         for section in re.split(r"\n\s*\n", text):
#             section = section.strip()

#             if not section:
#                 continue

#             if len(section) > size:
#                 if current:
#                     chunks.append(current)
#                     current = ""

#                 chunks.extend(
#                     section[i:i + size]
#                     for i in range(0, len(section), size)
#                 )

#             elif len(current) + len(section) + (2 if current else 0) <= size:
#                 current += (
#                     "\n\n" if current else ""
#                 ) + section

#             else:
#                 if current:
#                     chunks.append(current)

#                 current = section

#         if current:
#             chunks.append(current)

#     else:
#         raise ValueError(
#             "Choose 'fixed' or 'paragraph-aware'"
#         )

#     if not chunks:
#         raise ValueError("No text chunks produced")

#     # Step 4: Store prepared document records
#     target = out / "prepared_text.jsonl"

#     with target.open("w", encoding="utf-8") as handle:
#         for i, chunk in enumerate(chunks, 1):
#             record = {
#                 "id": i,
#                 "source": str(src),
#                 "text": chunk,
#                 "page": None
#             }
#             handle.write(
#                 json.dumps(record, ensure_ascii=False) + "\n"
#             )

#     # Step 5: Return source-to-output mapping
#     mapping = [
#         {
#             "source": str(src),
#             "output": str(target)
#         }
#     ]

#     processing_method = (
#         f"{method} text chunks of {size} characters"
#     )

#     return [target], mapping, processing_method

# # Step 6: Execute and register the pipeline
# run_A = register_and_run(
#     "documents",
#     SOURCE_A,
#     inspect_documents,
#     prepare_documents,
#     CONFIG_A
# )
# print("Cell 8A completed successfully.")


In [24]:
# # Cell 8B — CONDITIONAL REQUIRED: CSV/XLSX Projects
# # Requires Cell 7 — Common V1 Pipeline Utilities
# import pandas as pd
# # Step 1: Select source and configuration
# # Replace the dummy path with your actual dataset.
# SOURCE_B = resolve_source(
#     "topic3_dummy_datasets/02_tabular/student_performance_demo.csv"
# )

# # Use the same preparation settings selected in Part B.
# CONFIG_B = {
#     "remove_exact_duplicates": True,
#     "trim_text": True
# }
# # Step 2: Define a reusable table loader
# def read_table(src):
#     if src.suffix.lower() == ".csv":
#         return pd.read_csv(src)
#     elif src.suffix.lower() == ".xlsx":
#         return pd.read_excel(src)

#     else:
#         raise ValueError("Expected a CSV or XLSX source")
# # Step 3: Inspect the source dataset
# def inspect_table(inputs, cfg):
#     df = read_table(inputs[0])

#     if df.empty:
#         raise ValueError("Empty input table")

#     return {
#         "rows": len(df),
#         "columns": df.columns.tolist(),
#         "missing": {
#             str(k): int(v)
#             for k, v in df.isna().sum().items()
#         },
#         "exact_duplicates": int(df.duplicated().sum())
#     }
# # Step 4: Prepare the tabular dataset
# def prepare_table(inputs, out, cfg):
#     src = inputs[0]
#     df = read_table(src)
#     result = df.copy()
#     # Apply the preparation decisions from Part B
#     if cfg["remove_exact_duplicates"]:
#         result = result.drop_duplicates().copy()

#     if cfg["trim_text"]:
#         for col in result.select_dtypes(
#             include=["object", "string"]
#         ).columns:
#             result[col] = (
#                 result[col].astype("string").str.strip()
#             )
#     if result.empty:
#         raise ValueError("No prepared table rows")

#     # Step 5: Save prepared data
#     target = out / "prepared_table.csv"
#     result.to_csv(target, index=False)

#     # Step 6: Describe actual processing
#     method = (
#         f"remove_exact_duplicates={cfg['remove_exact_duplicates']}; "
#         f"trim_text={cfg['trim_text']}"
#     )
#     # Step 7: Return output and source-to-output mapping
#     mapping = [
#         {
#             "source": str(src),
#             "output": str(target)
#         }
#     ]
#     return [target], mapping, method

# # Step 8: Execute and register the selected pipeline
# run_B = register_and_run(
#     "tabular",
#     SOURCE_B,
#     inspect_table,
#     prepare_table,
#     CONFIG_B
# )
# print("Cell 8B completed successfully.")


In [28]:
# Cell 8C — CONDITIONAL REQUIRED: Computer Vision Projects
# Requires Cell 7 — Common V1 Pipeline Utilities

from PIL import Image

# Step 1: Select source folder and preparation settings
# Point directly to RAW_DIR using "." since your screenshots are at the root level.
SOURCE_C = resolve_source(
    ".",
    folder=True
)

# Match the configuration selected in Part B.
CONFIG_C = {
    "resize": [224, 224],
    "convert_rgb": True
}

# Step 2: Inspect source images
def inspect_images(inputs, cfg):
    properties = []
    for src in inputs:
        # Verify that the image can be opened
        with Image.open(src) as im:
            im.verify()
        # Reopen after verification to inspect properties
        with Image.open(src) as im:
            properties.append({
                "name": src.name,
                "size": list(im.size),
                "mode": im.mode
            })
    return {
        "image_count": len(inputs),
        "sample": properties[:3]
    }

# Step 3: Prepare images
def prepare_images(inputs, out, cfg):
    outputs = []
    mapping = []
    image_size = cfg["resize"]
    convert_rgb = cfg["convert_rgb"]
    for i, src in enumerate(inputs, 1):
        with Image.open(src) as im:
            # Apply selected colour conversion
            if convert_rgb:
                image = im.convert("RGB")
            else:
                image = im.copy()

            # Apply resizing only if selected in Part B
            if image_size is not None:
                image = image.resize(tuple(image_size))

            # Save prepared image
            target = out / f"prepared_{i:03d}.png"
            image.save(target)

        # Record output
        outputs.append(target)

        # Preserve source-to-output relationship
        mapping.append({
            "source": str(src),
            "output": str(target)
        })
    # Describe actual processing decisions
    processing_method = (
        f"convert_rgb={convert_rgb}; "
        f"resize={image_size}"
    )
    return outputs, mapping, processing_method

# Step 4: Execute and register the pipeline
run_C = register_and_run(
    "images",
    SOURCE_C,
    inspect_images,
    prepare_images,
    CONFIG_C
)
print("Cell 8C completed successfully.")

SUCCESS: images | Inputs: 4 | Outputs: 4
Manifest: /content/week3_project/data/pipeline_v1/images/pipeline_manifest.json
Cell 8C completed successfully.


In [ ]:
# # Cell 8D — CONDITIONAL REQUIRED: JSON/API Projects
# # Requires Cell 7 — Common V1 Pipeline Utilities
# import json
# # Step 1: Select source and configuration
# # Replace the dummy path with your actual dataset.

# SOURCE_D = resolve_source(
#     "topic3_dummy_datasets/04_json_api/products_api_demo.json"
# )
# # Match the settings selected in Part B.
# CONFIG_D = {
#     "record_key": "items",
#     "required_keys": ["item_id"],
#     "invalid_record_policy": "exclude"
# }

# # Step 2: Extract JSON records
# def extract_json_records(source, cfg):
#     with source.open("r", encoding="utf-8") as file:
#         obj = json.load(file)
#     if isinstance(obj, list):
#         records = obj
#     elif isinstance(obj, dict) and isinstance(
#         obj.get(cfg["record_key"]), list
#     ):
#         records = obj[cfg["record_key"]]
#     elif isinstance(obj, dict) and cfg["record_key"] not in obj:
#         records = [obj]
#     else:
#         raise ValueError(
#             "Expected a JSON list, a JSON object, or "
#             f"an object containing a list under '{cfg['record_key']}'."
#         )
#     if not records:
#         raise ValueError("No JSON records found.")
#     return records

# # Step 3: Inspect source structure
# def inspect_json(inputs, cfg):
#     records = extract_json_records(inputs[0], cfg)
#     object_records = [
#         record for record in records
#         if isinstance(record, dict)
#     ]
#     valid = [
#         record for record in object_records
#         if all(
#             key in record
#             for key in cfg["required_keys"]
#         )
#     ]
#     return {
#         "record_count": len(records),
#         "object_records": len(object_records),
#         "valid_records": len(valid),
#         "excluded_records": len(records) - len(valid),
#         "required_keys": cfg["required_keys"],
#         "first_keys": (
#             list(object_records[0].keys())
#             if object_records else []
#         )
#     }
# # Step 4: Prepare and export JSONL
# def prepare_json(inputs, out, cfg):
#     src = inputs[0]
#     records = extract_json_records(src, cfg)
#     policy = cfg["invalid_record_policy"]
#     if policy != "exclude":
#         raise ValueError(
#             "This V1 example supports only the 'exclude' policy."
#         )
#     valid = [
#         record for record in records
#         if isinstance(record, dict)
#         and all(
#             key in record
#             for key in cfg["required_keys"]
#         )
#     ]
#     excluded = len(records) - len(valid)
#     if not valid:
#         raise ValueError(
#             "No valid JSON records remain after validation."
#         )
#     target = out / "prepared_json.jsonl"
#     with target.open("w", encoding="utf-8") as handle:
#         for record in valid:
#             handle.write(
#                 json.dumps(record, ensure_ascii=False) + "\n"
#             )
#     # Basic validation: read back the exported JSONL
#     with target.open("r", encoding="utf-8") as handle:
#         exported = [
#             json.loads(line)
#             for line in handle
#             if line.strip()
#         ]
#     if exported != valid:
#         raise ValueError(
#             "Exported JSONL content does not match prepared records."
#         )
#     mapping = [
#         {
#             "source": str(src),
#             "output": str(target)
#         }
#     ]
#     method = (
#         f"JSON to JSONL; record_key={cfg['record_key']}; "
#         f"required_keys={cfg['required_keys']}; "
#         f"excluded_records={excluded}"
#     )
#     return [target], mapping, method
# # Step 5: Execute and register pipeline
# run_D = register_and_run(
#     "json",
#     SOURCE_D,
#     inspect_json,
#     prepare_json,
#     CONFIG_D
# )
# print("Cell 8D completed successfully.")
# print("Inspection:", run_D["inspection"])


In [ ]:
# # Cell 8E — CONDITIONAL REQUIRED: Audio Projects
# # Requires Cell 7 — Common V1 Pipeline Utilities

# !pip -q install soundfile

# import soundfile as sf
# import json

# # Step 1: Select source and configuration
# # Replace with your actual audio source if applicable.

# SOURCE_E = resolve_source(
#     "topic3_dummy_datasets/05_audio/tone_low.wav"
# )

# # Match the preparation decision from Part B.
# CONFIG_E = {
#     "method": "inspection-only"
# }


# # Step 2: Inspect audio properties
# def inspect_audio(inputs, cfg):
#     src = inputs[0]
#     info = sf.info(str(src))

#     if info.frames <= 0:
#         raise ValueError("Audio source has no frames")

#     if info.samplerate <= 0:
#         raise ValueError("Invalid sample rate")

#     if info.channels <= 0:
#         raise ValueError("Invalid channel count")

#     if info.duration <= 0:
#         raise ValueError("Invalid audio duration")

#     return {
#         "sample_rate": info.samplerate,
#         "channels": info.channels,
#         "duration_seconds": info.duration,
#         "frames": info.frames,
#         "format": info.format,
#         "subtype": info.subtype
#     }


# # Step 3: Prepare the audio inventory
# def prepare_audio(inputs, out, cfg):
#     src = inputs[0]

#     # Reuse the inspection function
#     audio_info = inspect_audio(inputs, cfg)

#     audio_record = {
#         "source": str(src),
#         "source_sha256": sha256_file(src),
#         "inspection": audio_info,
#         "processing": cfg["method"]
#     }
#     # Step 4: Save the inventory
#     target = out / "audio_inventory.json"
#     target.write_text(
#         json.dumps(
#             audio_record,
#             indent=2,
#             ensure_ascii=False
#         ),
#         encoding="utf-8"
#     )
#     # Step 5: Return source-to-output mapping
#     mapping = [
#         {
#             "source": str(src),
#             "output": str(target)
#         }
#     ]
#     return [target], mapping, cfg["method"]
# # Step 6: Execute and register the pipeline
# run_E = register_and_run(
#     "audio",
#     SOURCE_E,
#     inspect_audio,
#     prepare_audio,
#     CONFIG_E
# )

# print("Cell 8E completed successfully.")


In [ ]:
# # Cell 8F — CONDITIONAL REQUIRED: Time-Series Projects
# # Requires Cell 7 — Common V1 Pipeline Utilities

# import pandas as pd

# # Step 1: Select source and configuration
# # Replace with your actual dataset where applicable.

# SOURCE_F = resolve_source(
#     "topic3_dummy_datasets/06_timeseries/energy_readings_demo.csv"
# )

# # Use the same preparation settings selected in Part B.
# CONFIG_F = {
#     "time_column": "timestamp",
#     "drop_invalid_timestamps": True
# }


# # Step 2: Inspect source time-series
# def inspect_timeseries(inputs, cfg):
#     df = pd.read_csv(inputs[0])
#     col = cfg["time_column"]

#     if df.empty:
#         raise ValueError("Empty time-series dataset")

#     if col not in df.columns:
#         raise ValueError(
#             f"Missing timestamp column {col!r}; "
#             f"available columns: {df.columns.tolist()}"
#         )

#     parsed = pd.to_datetime(
#         df[col],
#         errors="coerce",
#         utc=True
#     )

#     return {
#         "rows": len(df),
#         "columns": df.columns.tolist(),
#         "invalid_timestamps": int(parsed.isna().sum()),
#         "duplicate_valid_timestamps": int(
#             parsed.dropna().duplicated().sum()
#         ),
#         "missing_by_column": {
#             str(k): int(v)
#             for k, v in df.isna().sum().items()
#         }
#     }


# # Step 3: Prepare time-series data
# def prepare_timeseries(inputs, out, cfg):
#     src = inputs[0]
#     df = pd.read_csv(src)
#     col = cfg["time_column"]

#     if col not in df.columns:
#         raise ValueError(
#             f"Missing timestamp column: {col}"
#         )

#     df[col] = pd.to_datetime(
#         df[col],
#         errors="coerce",
#         utc=True
#     )

#     # Apply the selected Part B decision
#     if cfg["drop_invalid_timestamps"]:
#         df = df.dropna(subset=[col]).copy()

#     # Sort chronologically
#     result = df.sort_values(
#         by=col,
#         na_position="last"
#     ).copy()

#     if result.empty:
#         raise ValueError(
#             "No time-series records remain after preparation"
#         )

#     # Step 4: Store prepared data
#     target = out / "prepared_timeseries.csv"
#     result.to_csv(target, index=False)

#     # Step 5: Validate exported dataset
#     exported = pd.read_csv(target)

#     if len(exported) != len(result):
#         raise ValueError(
#             "Exported row count does not match prepared data"
#         )

#     if exported.columns.tolist() != result.columns.tolist():
#         raise ValueError(
#             "Exported columns do not match prepared data"
#         )

#     exported_times = pd.to_datetime(
#         exported[col],
#         errors="coerce",
#         utc=True
#     ).dropna()
#     if not exported_times.is_monotonic_increasing:
#         raise ValueError(
#             "Exported timestamps are not chronologically sorted"
#         )
#     # Step 6: Record source-to-output relationship
#     method = (
#         f"time_column={col}; "
#         f"drop_invalid_timestamps="
#         f"{cfg['drop_invalid_timestamps']}; "
#         "chronological_sort=True"
#     )
#     mapping = [
#         {
#             "source": str(src),
#             "output": str(target)
#         }
#     ]
#     return [target], mapping, method

# # Step 7: Execute and register pipeline
# run_F = register_and_run(
#     "timeseries",
#     SOURCE_F,
#     inspect_timeseries,
#     prepare_timeseries,
#     CONFIG_F
# )
# print("Cell 8F completed successfully.")


In [29]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")


--- Repeatability Verification ---
SUCCESS: images | Inputs: 4 | Outputs: 4
Manifest: /content/week3_project/data/pipeline_v1/images/pipeline_manifest.json
PASS: images — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
